In [ ]:
#导入相关库
import os
import cv2
import torch
import clip
from PIL import Image
import pandas as pd
from tqdm import tqdm
import shutil

In [ ]:
#计算视频文本相似度函数
def process_video(video_path, text_content, model, preprocess, device, fps=1):
    """按秒提取视频帧并计算CLIP相似度"""
    cap = cv2.VideoCapture(video_path)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    text_input = clip.tokenize([text_content], truncate=True).to(device)
    
    with torch.no_grad():
        text_features = model.encode_text(text_input)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    
    scores = []
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # 按秒采样（假设fps=1表示每秒取1帧）
        if frame_count % int(frame_rate / fps) == 0:
            # 转换帧为PIL Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            # CLIP处理
            image_input = preprocess(pil_image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                image_features = model.encode_image(image_input)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                similarity = (image_features @ text_features.T).item()
                
            scores.append(similarity)
            
        frame_count += 1
    
    cap.release()
    return scores

In [ ]:
#设置GPU 导入模型
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
#temp_path视频路径；text_content视频标题或视频介绍文本
scores = process_video(temp_path, text_content, model, preprocess, device)